![](https://github.com/quarcs-lab/project2021o-notebook/blob/main/figs/cover.png?raw=true)

```
https://shorturl.at/evEFS
```
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/bolivia112/blob/main/notebooks/eda_esda.ipynb)

# **A geocomputational notebook to monitor regional development in Bolivia**

Carlos Mendez (Nagoya Univerisity), Erick Gonzales (United Nations), Lykke Andersen (SDSN Bolivia)

This notebook analyzes regional development across Bolivia's 112 provinces. Province values are population-weighted aggregations of the underlying municipal data (intensive variables = weighted mean, extensive variables = sum); see [../province_aggregation_report.md](../province_aggregation_report.md) for the aggregation methodology.


[![DOI](https://zenodo.org/badge/683583423.svg)](https://zenodo.org/badge/latestdoi/683583423)
```
Suggested citation:
Mendez, C., Gonzales, E., & Andersen, L. (2023). A geocomputational notebook to monitor regional development in Bolivia. Zenodo. https://doi.org/10.5281/zenodo.828685
```
Github repository: https://github.com/quarcs-lab/project2021o-notebook

## 1) Setup

In [ ]:
# Install packages not included in Google Colab's default environment.
# When running locally with UV, these are already installed and this cell is harmless.
!pip install geopandas contextily libpysal esda splot mapclassify inequality mgwr -q

In [ ]:
# Importing necessary libraries for data analysis and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

# Importing libraries for spatial data and visualization
import geopandas as gpd
import contextily as cx

import libpysal
from libpysal  import weights
from libpysal.weights import Queen

# Exploratory Spatial Data Analysis (ESDA) tools
import mapclassify as mc
import esda
from esda.moran import Moran, Moran_Local

# Spatial plotting tools
import splot
from splot.esda import moran_scatterplot, plot_moran, lisa_cluster, plot_local_autocorrelation
from splot.libpysal import plot_spatial_weights
from splot.mapping import vba_choropleth

# Library for inequality analysis
import inequality
from inequality.gini import Gini_Spatial

# Statistical modeling
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Geographically Weighted Regression (GWR) and related utilities
from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW
from mgwr.utils import shift_colormap, truncate_colormap

# Suppressing warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

## 2) Import data

In [ ]:
# Province boundaries (112 provinces) provide the geometry plus the
# COORD_X/COORD_Y centroids used later for GWR/MGWR.
# Prefer the local repository path; the raw-GitHub URL is provided as a fallback.
mapURL = 'https://raw.githubusercontent.com/quarcs-lab/bolivia112/main/maps/bolivia112provincesOpt.geojson'
# Local path (uncomment when running from the repository root):
# mapURL = 'maps/bolivia112provincesOpt.geojson'

# Province-level attribute table (all indicators keyed by prov_id). Province
# values are population-weighted aggregations of the municipal data (intensive
# vars = weighted mean, extensive vars = sum); see
# ../province_aggregation_report.md for the full methodology.
dataURL = 'https://raw.githubusercontent.com/quarcs-lab/bolivia112/main/bolivia112_v20260622.csv'
# Local path (uncomment when running from the repository root):
# dataURL = 'bolivia112_v20260622.csv'

# Read the province boundaries with GeoPandas and the attributes with pandas.
gdf_map = gpd.read_file(mapURL)
df_data = pd.read_csv(dataURL)

# Drop admin columns that also live in the attribute table to avoid merge
# collisions; keep geometry and the COORD_X/COORD_Y centroids from the map.
gdf_map = gdf_map.drop(columns=['prov', 'dep', 'dep_id', 'shapeName'])

# Merge boundaries and attributes on the province join key prov_id
gdf = gdf_map.merge(df_data, on='prov_id', how='inner')

In [ ]:
gdf.head(3)

In [ ]:
dataDefinitions = pd.read_csv('https://raw.githubusercontent.com/quarcs-lab/project2021o-notebook/main/dataDefinitions.csv')
dataDefinitions

In [ ]:
data_dict = dict(zip(dataDefinitions['Variable'], dataDefinitions['Label']))

## 3) Define key parameters

In [ ]:
INDICATOR1 = 'imds'

In [ ]:
INDICATOR2 = 'pop2017'

In [ ]:
INDICATOR3 = 'ln_t400NTLpc2017'

In [ ]:
INDICATOR4 = 'rank_imds'

In [ ]:
INDICATOR5 = 'co2017'

In [ ]:
ADM1 = 'dep'                       # Department (9 departments)

In [ ]:
ADM2 = 'prov'                      # Province (112 provinces)

In [ ]:
# Unit of analysis: the province (112 provinces). The municipal-level ADM3
# of the source notebook is replaced by the province here.
ADM3 = 'prov'

## 4) Exploratory data analysis (EDA)

In [ ]:
# Exploring data using the explore() function on the GeoDataFrame
gdf.explore(
    column=ADM1,                        # Column to visualize
    tooltip=[ADM1, ADM3, INDICATOR1, INDICATOR4],  # Information in tooltip
    categorical=True,                    # Categorical column
    legend=True,                         # Show legend
    tiles='CartoDB positron',            # Basemap style (Other options: CartoDB positron, OpenStreetMap, Stamen Terrain)
    style_kwds=dict(color="gray", weight=0.6),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)     # Customize legend
)

In [ ]:
gdf[[ADM3, ADM1, INDICATOR1, INDICATOR2, INDICATOR3, INDICATOR4]].sort_values(by=INDICATOR1, ascending=False).reset_index(drop=True)

### 4.1) Descriptive statistics

- What do we know about the centrality and dispersion of our development indicator?

In [ ]:
gdf[[INDICATOR1, INDICATOR2, INDICATOR3, INDICATOR4]].describe().round(2)

In [ ]:
gdf[[ADM1, INDICATOR1]].groupby('dep').describe().round(2)

### 4.2) Regional development differences

- How is our development indicator disbributed across the 112 provinces and 9 departments of Bolivia?

In [ ]:
# Creating a strip plot using Plotly Express
px.strip(
    gdf,
    x=INDICATOR1,                 # Data for x-axis
    y=ADM1,                      # Data for y-axis
    color=ADM1,                  # Color grouping based on 'dep'
    hover_name=ADM3,             # Province for hover tooltip
    hover_data=[ADM1, INDICATOR4],     # Additional data for hover tooltip
    labels=dict(data_dict)        # Custom labels from data_dict
)

In [ ]:
# Creating a histogram using Plotly Express
px.histogram(
    gdf,
    x=INDICATOR1,                    # Data for x-axis
    color=ADM1,                     # Color grouping based on 'dep'
    hover_name=ADM3,                # Province for hover tooltip
    marginal='rug',                  # Display rug plot on the marginal axis
    hover_data=[ADM1, INDICATOR4],        # Additional data for hover tooltip
    labels=dict(data_dict)           # Custom labels from data_dict
)

In [ ]:
# Creating a box plot using Plotly Express
px.box(
    gdf,
    x=INDICATOR1,                 # Data for x-axis
    y=ADM1,                      # Data for y-axis
    color=ADM1,                  # Color grouping based on 'dep'
    hover_name=ADM3,             # Province for hover tooltip
    hover_data=[ADM1, INDICATOR4],     # Additional data for hover tooltip
    labels=dict(data_dict)        # Custom labels from data_dict
)

### 4.3) Population and development

- Do the most populous regions tend to be more developed?

In [ ]:
px.bar(
    gdf.sort_values(by=INDICATOR2, ascending=True),
    x=ADM3,                    # Data for x-axis
    y=INDICATOR2,               # Data for y-axis
    log_y=False,                # Disable logarithmic y-axis scale
    hover_data=[INDICATOR1, INDICATOR2, INDICATOR4],  # Additional data for hover tooltip
    labels=dict(data_dict)         # Custom labels from data_dict
)

In [ ]:
# Creating a treemap using Plotly Express
px.treemap(
    gdf,
    color=INDICATOR1,                # Data for coloring
    values=INDICATOR2,                # Data for size of treemap blocks
    path=[ADM1, ADM3],             # Path for hierarchical display
    hover_name= ADM3,                # Province for hover tooltip
    hover_data=[INDICATOR4],        # Additional data for hover tooltip
    labels=dict(data_dict)           # Custom labels from data_dict
)

In [ ]:
# Creating a sunburst plot using Plotly Express
px.sunburst(
    gdf,
    color=INDICATOR1,                # Data for coloring
    values=INDICATOR2,                # Data for size of sunburst segments
    path=[ADM1, ADM3],             # Path for hierarchical display
    hover_name=ADM3,                # Province for hover tooltip
    hover_data=[INDICATOR4],        # Additional data for hover tooltip
    labels=dict(data_dict)           # Custom labels from data_dict
)

In [ ]:
px.scatter(
    gdf,
    x=INDICATOR2,                             # Data for x-axis
    log_x = True,
    y=INDICATOR1,                             # Data for y-axis
    color=ADM1,                              # Data for color coding
    symbol=ADM1,                             # Data for symbol coding
    hover_name=ADM3,                         # Province for hover tooltip
    trendline='ols',                          # Adding OLS trendline
    hover_data=[INDICATOR4],                 # Additional data for hover tooltip
    labels=dict(data_dict)                    # Custom labels from data_dict
)

### 4.4) Nighttime lights and development

- Can nighttime lights help us predict regional development?

In [ ]:
# Exploring data using the explore() function on the GeoDataFrame
gdf.explore(
    column=INDICATOR1,                 # Column to visualize
    tooltip=[ADM1, ADM3, INDICATOR1, INDICATOR4],  # Information to show in tooltip
    k=3,                               # Number of classes for visualization
    scheme='FisherJenks',              # Classification scheme (info: https://pysal.org/mapclassify/generated/mapclassify.FisherJenks.html#mapclassify.FisherJenks)
    cmap='BuPu',                   # Color map for visualization
    legend=True,                       # Show legend
    tiles= cx.providers.NASAGIBS.ViirsEarthAtNight2012,           # Basemap style (other options: https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html)
    style_kwds=dict(color="gray", weight=0.4, alpha=0.9),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)   # Legend customization
)

In [ ]:
# Creating a scatter plot with Plotly Express
px.scatter(
    gdf,
    x=INDICATOR3,                            # Data for x-axis
    y=INDICATOR1,                            # Data for y-axis
    color=ADM1,                             # Data for color coding
    symbol=ADM1,                            # Data for symbol coding
    hover_name=ADM3,                        # Province for hover tooltip
    trendline='ols',                         # Adding OLS trendline
    trendline_scope='overall',               # Scope of the trendline (overall)
    hover_data=[INDICATOR3],                # Additional data for hover tooltip
    labels=dict(data_dict)                   # Custom labels from data_dict
)

How heterogeneious is the relatinship between regional development and nighttime lights?

In [ ]:
# Creating a scatter plot with marginal box plots using Plotly Express
px.scatter(
    gdf,
    x=INDICATOR3,                             # Data for x-axis
    y=INDICATOR1,                             # Data for y-axis
    color=ADM1,                              # Data for color coding
    symbol=ADM1,                             # Data for symbol coding
    hover_name=ADM3,                         # Province for hover tooltip
    trendline='ols',                          # Adding OLS trendline
    marginal_x="box",                         # Display marginal box plot on x-axis
    marginal_y="box",                         # Display marginal box plot on y-axis
    hover_data=[INDICATOR4],                 # Additional data for hover tooltip
    labels=dict(data_dict)                    # Custom labels from data_dict
)

## 5) Exploratory spatial data analysis (ESDA)

### 5.1) Spatial distribution

#### 5.1.1) Box-plot breaks

In [ ]:
# Creating a box plot using Plotly Express
px.box(
    gdf,
    x=INDICATOR1,                 # Data for x-axis
    #y=ADM1,                      # Data for y-axis
    #color=ADM1,                  # Color grouping based on ADM1
    hover_name=ADM3,             # Province for hover tooltip
    hover_data=[INDICATOR4, ADM1],     # Additional data for hover tooltip
    labels=dict(data_dict)        # Custom labels from data_dict
)

In [ ]:
# Exploring data using the explore() function on the GeoDataFrame
gdf.explore(
    column=INDICATOR1,                 # Column to visualize
    tooltip=[ADM1, ADM3, INDICATOR1, INDICATOR4],  # Information to show in tooltip
    scheme='BoxPlot',              # Classification scheme (info: https://pysal.org/mapclassify/generated/mapclassify.BoxPlot.html#mapclassify.BoxPlot)
    k=6,                            # Number of classes for coloring
    cmap='coolwarm',                   # Color map for visualization
    legend=True,                       # Show legend
    tiles='CartoDB positron',            # Basemap style (other options: https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html)
    style_kwds=dict(color="gray", weight=0.4),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)   # Legend customization
)

In [ ]:
mc.BoxPlot(gdf[INDICATOR1])

In [ ]:
import contextily as cx

# Create a figure and axis for plotting
fig, ax = plt.subplots(figsize=(9, 6))

# Plot the GeoDataFrame with specified settings
gdf.plot(
    column=INDICATOR1,              # Data for coloring
    scheme='BoxPlot',           # Classification scheme
    cmap='coolwarm',                # Color map for visualization
    edgecolor='k',                  # Color of edges
    linewidth=0.5,                  # Width of edge lines
    alpha=0.8,                      # Transparency
    legend=True,                    # Show legend
    ax=ax,                          # Specify the axis for plotting
    legend_kwds={'bbox_to_anchor': (1.00, 0.92)}  # Legend customization
)

# Adding a basemap using contextily
cx.add_basemap(
    ax,                             # Axis to add basemap to
    crs=gdf.crs.to_string(),         # Coordinate reference system
    source=cx.providers.Stamen.TonerHybrid,  # Basemap source
    attribution=False               # Disable attribution
)

# Set plot title and adjust layout
plt.title('Spatial distribution: Three natural breaks')
plt.tight_layout()

# Turn off axis display
ax.axis("off")

# Display the plot
plt.show()

#### 5.1.2) Fisher-Jenks breaks

In [ ]:
# Exploring data using the explore() function on the GeoDataFrame
gdf.explore(
    column=INDICATOR1,                 # Column to visualize
    tooltip=[ADM1, ADM3, INDICATOR1, INDICATOR4],  # Information to show in tooltip
    k=3,                               # Number of classes for visualization
    scheme='FisherJenks',              # Classification scheme (info: https://pysal.org/mapclassify/generated/mapclassify.FisherJenks.html#mapclassify.FisherJenks)
    cmap='coolwarm',                   # Color map for visualization
    legend=True,                       # Show legend
    tiles='CartoDB positron',            # Basemap style (other options: https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html)
    style_kwds=dict(color="gray", weight=0.4),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)   # Legend customization
)

In [ ]:
mc.FisherJenks(gdf[INDICATOR1], k=3)

In [ ]:
import contextily as cx

# Create a figure and axis for plotting
fig, ax = plt.subplots(figsize=(9, 6))

# Plot the GeoDataFrame with specified settings
gdf.plot(
    column=INDICATOR1,              # Data for coloring
    scheme='FisherJenks',           # Classification scheme
    k=3,                            # Number of classes for coloring
    cmap='coolwarm',                # Color map for visualization
    edgecolor='k',                  # Color of edges
    linewidth=0.5,                  # Width of edge lines
    alpha=0.8,                      # Transparency
    legend=True,                    # Show legend
    ax=ax,                          # Specify the axis for plotting
    legend_kwds={'bbox_to_anchor': (1.00, 0.92)}  # Legend customization
)

# Adding a basemap using contextily
cx.add_basemap(
    ax,                             # Axis to add basemap to
    crs=gdf.crs.to_string(),         # Coordinate reference system
    source=cx.providers.Stamen.TonerHybrid,  # Basemap source
    attribution=False               # Disable attribution
)

# Set plot title and adjust layout
plt.title('Spatial distribution: Three natural breaks')
plt.tight_layout()

# Turn off axis display
ax.axis("off")

# Display the plot
plt.show()

### 5.2) Spatial dependence

- To what extent is the performance of a region similar to that of its neighbours?
- To what extent is there is an overall pattern of "clustering"?
- Where can we find statistically significant spatial clusters?
- Where can we find statistically significant spatial outliers?

#### 5.2.1) Spatial connectivity

In [ ]:
# Create K-nearest neighbors spatial weights
W = weights.KNN.from_dataframe(gdf, k=6)

# Transform the weights to row-standardized
W.transform = 'r'

# Plot the spatial weights using splot
plot_spatial_weights(W, gdf)

#### 5.2.2) Global dependence

In [ ]:
# List of columns to select from the GeoDataFrame
myLIST = [ADM1, ADM3, INDICATOR1]
# Create a new DataFrame with selected columns
df_MORAN = gdf[myLIST]
# Calculate spatial lag of INDICATOR1 using the specified weights
df_MORAN["WxINDICATOR1"] = weights.lag_spatial(W, df_MORAN.iloc[: , -1])

In [ ]:
# Creating a scatter plot with Plotly Express
px.scatter(
    df_MORAN,
    x=INDICATOR1,                            # Data for x-axis
    y="WxINDICATOR1",                        # Data for y-axis
    hover_name=ADM3,                        # Province for hover tooltip
    hover_data=[ADM1, INDICATOR1, 'WxINDICATOR1'],  # Additional data for hover tooltip
    trendline="ols",                         # Adding OLS trendline
    marginal_x="box",                        # Display marginal box plot on x-axis
    marginal_y="box",                        # Display marginal box plot on y-axis
    labels=dict(data_dict)                    # Custom labels from data_dict
)

#### 5.2.3) Local dependence

In [ ]:
# Calculate global Moran's I
globalMoran = Moran(gdf[INDICATOR1], W)
MoranI = globalMoran.I
MoranI = "{:.2f}".format(MoranI)

# Calculate local indicators of spatial association
localMoran = Moran_Local(gdf[INDICATOR1], W, permutations = 999, seed=12345)

In [ ]:
# Create subplots for visualizations
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))

# Plotting Moran scatterplot
moran_scatterplot(localMoran, p=0.10, aspect_equal=False, zstandard=False, ax=axes[0])

# Plotting Local Indicators of Spatial Association (LISA) clusters
lisa_cluster(localMoran, gdf, p=0.10, legend_kwds={'bbox_to_anchor': (0.02, 0.90)}, ax=axes[1])

# Setting labels and titles for the plots
axes[0].set_xlabel('Indicator')
axes[0].set_ylabel('Spatial lag of indicator')
axes[0].set_title(f"(a) Moran scatterplot (Moran's I = {MoranI})")
axes[1].set_title("(b) Spatial clusters and outliers (p < 0.10)")

# Display the plots
plt.show()

### 5.3) Spatial inequality

In [ ]:
# Create a histogram plot using Seaborn
sns.histplot(x=gdf[INDICATOR1], kde=True);

In [ ]:
# Create a histogram plot using Seaborn
sns.histplot(x=gdf[INDICATOR3], kde=True);

#### 5.3.1) Theil index

In [ ]:
theil_INDICATOR1 = inequality.theil.Theil(gdf[INDICATOR1].values).T
theil_INDICATOR1

In [ ]:
theil_INDICATOR3 = inequality.theil.Theil(gdf[INDICATOR3].values).T
theil_INDICATOR3

Theil index decomposition

In [ ]:
# Exploring data using the explore() function on the GeoDataFrame
gdf.explore(
    column=ADM1,                        # Column to visualize
    tooltip=[ADM1, ADM3, INDICATOR1, INDICATOR4],  # Information in tooltip
    categorical=True,                    # Categorical column
    legend=True,                         # Show legend
    tiles='CartoDB positron',            # Basemap style
    style_kwds=dict(color="gray", weight=0.6),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)     # Customize legend

)

In [ ]:
theil_BW_INDICATOR1 = inequality.theil.TheilD(gdf[INDICATOR1].values, gdf[ADM1])

In [ ]:
theil_B_INDICATOR1 = theil_BW_INDICATOR1.bg
theil_W_INDICATOR1 = theil_BW_INDICATOR1.wg

In [ ]:
theil_B_INDICATOR1/theil_INDICATOR1

In [ ]:
theil_W_INDICATOR1/theil_INDICATOR1

In [ ]:
theil_BW_INDICATOR3 = inequality.theil.TheilD(gdf[INDICATOR3].values, gdf[ADM1])

In [ ]:
theil_B_INDICATOR3 = theil_BW_INDICATOR3.bg
theil_W_INDICATOR3 = theil_BW_INDICATOR3.wg

In [ ]:
theil_B_INDICATOR3/theil_INDICATOR3

In [ ]:
theil_W_INDICATOR3/theil_INDICATOR3

#### 5.3.2) Gini index

In [ ]:
inequality.gini.Gini(gdf[INDICATOR1].values).g

In [ ]:
inequality.gini.Gini(gdf[INDICATOR3].values).g

Let's compute the spatial gini index

In [ ]:
Gini_Spatial(gdf[INDICATOR1], W).wcg_share

In [ ]:
Gini_Spatial(gdf[INDICATOR1], W).p_sim

In [ ]:
Gini_Spatial(gdf[INDICATOR3], W).wcg_share

In [ ]:
Gini_Spatial(gdf[INDICATOR3], W).p_sim

### 5.4) Spatial heterogeneity

In [ ]:
# Reshape the data for GWR regression
y = gdf[INDICATOR1].values.reshape((-1,1))
y.shape

In [ ]:
# Reshape the data for GWR regression
X = gdf[INDICATOR3].values.reshape((-1,1))
X.shape

In [ ]:
# Create coordinate pairs
u = gdf['COORD_X']
v = gdf['COORD_Y']
coords = list(zip(u,v))

#### 5.4.1) GWR analysis

In [ ]:
# Perform bandwidth selection for Geographically Weighted Regression (GWR)
gwr_selector = Sel_BW(coords, y, X, spherical = True)
gwr_bw = gwr_selector.search(criterion='AICc')

In [ ]:
# Print bandwidth interval
print('GWR bandwidth =', gwr_bw)

In [ ]:
# Perform Geographically Weighted Regression (GWR) and get results
gwr_results = GWR(coords, y, X, gwr_bw).fit()

# Display summary of GWR results
gwr_results.summary()

In [ ]:
# Print bandwidth interval
gwr_bw_ci = gwr_results.get_bws_intervals(gwr_selector)
print(gwr_bw_ci)

In [ ]:
# As reference, here is the (mean) R2, AIC, and AICc
print('Mean R2 =', gwr_results.R2)
print('AIC =',     gwr_results.aic)
print('AICc =',    gwr_results.aicc)

In [ ]:
# Add R2 to GeoDataframe
gdf['gwr_R2'] = gwr_results.localR2

In [ ]:
# Visualizing local R-squared using the explore() function
gdf.explore(
    column='gwr_R2',                   # Column to visualize
    tooltip=[ADM1, ADM3, 'gwr_R2',   # Information to show in tooltip
             INDICATOR4, INDICATOR1, INDICATOR3],  # Additional attributes
    k=5,                               # Number of classes for visualization
    scheme='FisherJenks',              # Classification scheme
    cmap='coolwarm',                   # Color map for visualization
    legend=True,                       # Show legend
    tiles='CartoDB dark_matter',       # Basemap style
    style_kwds=dict(color="gray", weight=0.4),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)   # Legend customization
)

In [ ]:
# Add coefficients to data frame
gdf['gwr_intercept'] = gwr_results.params[:,0]
gdf['gwr_slope1']     = gwr_results.params[:,1]

In [ ]:
# Filter t-values: standard alpha = 0.05
gwr_filtered_t1 = gwr_results.filter_tvals(alpha = 0.05)

In [ ]:
# Filter t-values: corrected alpha due to multiple testing
gwr_filtered_tc1 = gwr_results.filter_tvals()

In [ ]:
# Slope heterogeneity exploration using geopandas' explore() function
gdf.explore(
    column='gwr_slope1',                   # Column to visualize
    tooltip=['dep', 'prov', 'gwr_slope1',   # Information to show in tooltip
             'rank_imds', INDICATOR1, INDICATOR3],  # Additional attributes
    k=5,                                  # Number of classes for visualization
    scheme='FisherJenks',                 # Classification scheme
    cmap='coolwarm',                      # Color map for visualization
    legend=True,                          # Show legend
    tiles='CartoDB dark_matter',          # Basemap style
    style_kwds=dict(color="gray", weight=0.4),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)      # Legend customization
)

In [ ]:
# Create subplots for visualizations
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 6))

# Plotting the first map
gdf.plot(column='gwr_slope1', cmap='coolwarm', linewidth=0.01, scheme='FisherJenks', k=5,
         legend=True, legend_kwds={'bbox_to_anchor': (1.10, 0.96)}, ax=axes[0])
axes[0].set_title('(a) GWR: Nighttime lights (BW: ' + str(gwr_bw) + '), all coeffs', fontsize=12)
axes[0].axis("off")

# Plotting the second map
gdf.plot(column='gwr_slope1', cmap='coolwarm', linewidth=0.05, scheme='FisherJenks', k=5,
         legend=False, legend_kwds={'bbox_to_anchor': (1.10, 0.96)}, ax=axes[1])
gdf[gwr_filtered_t1[:, 1] == 0].plot(color='white', linewidth=0.05, edgecolor='black', ax=axes[1])
axes[1].set_title('(b) GWR: Nighttime lights (BW: ' + str(gwr_bw) + '), significant coeffs', fontsize=12)
axes[1].axis("off")

# Plotting the third map
gdf.plot(column='gwr_slope1', cmap='coolwarm', linewidth=0.05, scheme='FisherJenks', k=5,
         legend=False, legend_kwds={'bbox_to_anchor': (1.10, 0.96)}, ax=axes[2])
gdf[gwr_filtered_tc1[:, 1] == 0].plot(color='white', linewidth=0.05, edgecolor='black', ax=axes[2])
axes[2].set_title('(c) GWR: Nighttime lights (BW: ' + str(gwr_bw) + '), significant coeffs and corr. p-values', fontsize=12)
axes[2].axis("off")

# Adjust layout and display the plots
plt.tight_layout()
plt.show()

#### 5.4.2) MGWR analysis

In [ ]:
# Standardize variable for scale comparison
Zy = (y - y.mean(axis=0)) / y.std(axis=0)
ZX = (X - X.mean(axis=0)) / X.std(axis=0)

In [ ]:
# Perform bandwidth selection for Multiscale Geographically Weighted Regression (MGWR)
mgwr_selector = Sel_BW(coords, Zy, ZX, multi=True, spherical = True)
mgwr_bw = mgwr_selector.search(criterion='AICc')
mgwr_bw

In [ ]:
# Fit MGWR model
mgwr_results = MGWR(coords, Zy, ZX, mgwr_selector).fit()
mgwr_results.summary()

In [ ]:
#Show bandwidth intervals
mgwr_bw_ci = mgwr_results.get_bws_intervals(mgwr_selector)
print(mgwr_bw_ci)

In [ ]:
# As reference, here is the (mean) R2, AIC, and AICc
print('Mean R2 =', mgwr_results.R2)
print('AIC =',     mgwr_results.aic)
print('AICc =',    mgwr_results.aicc)

In [ ]:
# Add coefficients to data frame
gdf['mgwr_intercept'] = mgwr_results.params[:,0]
gdf['mgwr_slope1']     = mgwr_results.params[:,1]

In [ ]:
# Filter t-values: standard alpha = 0.05
mgwr_filtered_t1 = mgwr_results.filter_tvals(alpha = 0.05)

In [ ]:
# Filter t-values: corrected alpha due to multiple testing
mgwr_filtered_tc1 = mgwr_results.filter_tvals()

In [ ]:
# Slope heterogeneity exploration using geopandas' explore() function
gdf.explore(
    column='mgwr_slope1',                  # Column to visualize
    tooltip=[ADM1, ADM3, 'mgwr_slope1',  # Information to show in tooltip
             INDICATOR4, INDICATOR1, INDICATOR3],  # Additional attributes
    k=5,                                  # Number of classes for visualization
    scheme='FisherJenks',                 # Classification scheme
    cmap='coolwarm',                      # Color map for visualization
    legend=True,                          # Show legend
    tiles='CartoDB dark_matter',          # Basemap style
    style_kwds=dict(color="gray", weight=0.4),  # Styling for non-highlighted areas
    legend_kwds=dict(colorbar=False)      # Legend customization
)

In [ ]:
# Create subplots for visualizations
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 6))

# Plotting the first map
gdf.plot(column='mgwr_slope1', cmap='coolwarm', linewidth=0.01, scheme='FisherJenks', k=5,
         legend=True, legend_kwds={'bbox_to_anchor': (1.10, 0.96)}, ax=axes[0])
axes[0].set_title('(a) MGWR: Nighttime lights (BW: ' + str(mgwr_bw) + '), all coeffs', fontsize=12)
axes[0].axis("off")

# Plotting the second map
gdf.plot(column='mgwr_slope1', cmap='coolwarm', linewidth=0.05, scheme='FisherJenks', k=5,
         legend=False, legend_kwds={'bbox_to_anchor': (1.10, 0.96)}, ax=axes[1])
gdf[mgwr_filtered_t1[:, 1] == 0].plot(color='white', linewidth=0.05, edgecolor='black', ax=axes[1])
axes[1].set_title('(b) MGWR: Nighttime lights (BW: ' + str(mgwr_bw) + '), significant coeffs', fontsize=12)
axes[1].axis("off")

# Plotting the third map
gdf.plot(column='mgwr_slope1', cmap='coolwarm', linewidth=0.05, scheme='FisherJenks', k=5,
         legend=False, legend_kwds={'bbox_to_anchor': (1.10, 0.96)}, ax=axes[2])
gdf[mgwr_filtered_tc1[:, 1] == 0].plot(color='white', linewidth=0.05, edgecolor='black', ax=axes[2])
axes[2].set_title('(c) MGWR: Nighttime lights (BW: ' + str(mgwr_bw) + '), significant coeffs and corr. p-values', fontsize=12)
axes[2].axis("off")

# Adjust layout and display the plots
plt.tight_layout()
plt.show()